In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

file_path = "datasets/한글맞춤법 표준어규정 해설.pdf"

loader = PyPDFLoader(file_path)
pages = []

async for page in loader.alazy_load():
    pages.append(page)

/var/folders/9w/f6_frtc17sd4lj2jf_3h9qlm0000gn/T/ipykernel_66386/2924649121.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
print("페이지 수:", len(pages))
print("첫 번째 페이지 정보: ", pages[2])

페이지 수: 264
첫 번째 페이지 정보:  page_content='1988년 ‘한글 맞춤법’과 ‘표준어 규정’이 개정되면서, 국립국어원의 전신이라 
할 수 있는 국어연구소에서는 ‘한글 맞춤법 해설’, ‘표준어 규정 해설’(이하 ‘해
설’)을 함께 내놓았습니다. ‘해설’은 ‘한글 맞춤법’이나 ‘표준어 규정’의 내용을 
알기 쉽게 설명하면서, 동시에 규정 본문에서는 상세히 다루기 어려웠던 부분
을 보완하는 역할도 함께하였습니다. 또한 규정의 역사적 배경이나 관련 사항 
등도 담아 규정을 이해하는 데에 도움을 주고자 하였습니다. 실제로 많은 사람
이 ‘해설’과 규정을 거의 동등한 지위에 있는 것으로 인식할 만큼 지난 30년간 
‘해설’은 중요하게 다루어져 왔습니다.
그러나 시간이 지나 말이 변하면서 ‘해설’의 내용도 변할 수밖에 없는 부분이 
생겨났습니다. 먼저 1999년 “표준국어대사전”을 발간하면서 규정에는 명시되지 
않았던 맞춤법이나 표준어 관련 세부 사항을 사전 항목으로 담았습니다. 이는 
실질적으로 규정을 확장한 것으로 볼 수 있는데 이 과정에서 기존 ‘해설’과는 
달리 처리한 부분도 생겨났습니다. 또한 2011년 이후 국민의 언어생활 편의 증
진을 위해 많이 사용하는 비표준어나 방언을 국어심의회에서 표준어로 인정함
에 따라 기존 규정과 다른 부분이 생겼습니다. 이러한 내용을 담아 2017년에 
‘한글 맞춤법’과 ‘표준어 규정’을 일부 개정하였는데, 비록 규정의 방향이나 내
용은 크게 달라지지 않았지만 수정된 내용을 해설에 반영해야 하는 상황이 되
었습니다. 이에 국어연구소에서 펴낸 기존 ‘해설’을 보완하여 ‘국립국어원 해설’
을 발간하게 되었습니다.
머리말' metadata={'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2018-12-20T09:36:29+09:00', 'author': 'admin', 'm

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(pages)

In [4]:
print(f"총 {len(docs)}개의 청크 생성 완료")
print("각 청크의 길이:", [len(i.page_content) for i in docs])

for i in docs:
    print("[메타데이터]", i.metadata)
    print("[내용]", i.page_content)
    print("=" * 100)

총 510개의 청크 생성 완료
각 청크의 길이: [63, 467, 367, 472, 96, 473, 110, 343, 458, 471, 499, 129, 484, 456, 461, 124, 11, 420, 448, 448, 466, 463, 220, 489, 486, 484, 466, 54, 454, 476, 288, 468, 167, 468, 475, 452, 79, 452, 217, 474, 494, 268, 476, 332, 480, 242, 450, 451, 450, 202, 497, 259, 486, 482, 119, 491, 124, 458, 429, 456, 415, 492, 106, 479, 250, 489, 462, 102, 489, 387, 486, 356, 459, 432, 481, 254, 308, 345, 490, 304, 362, 493, 153, 476, 307, 473, 475, 50, 465, 195, 406, 460, 78, 491, 495, 52, 493, 213, 491, 254, 476, 144, 430, 498, 191, 453, 163, 500, 62, 479, 417, 473, 101, 497, 68, 486, 182, 477, 338, 490, 92, 473, 494, 101, 498, 48, 486, 489, 105, 461, 411, 469, 238, 485, 461, 223, 487, 190, 459, 469, 134, 462, 244, 457, 220, 478, 405, 485, 191, 464, 315, 482, 186, 487, 393, 483, 499, 486, 181, 483, 480, 481, 143, 460, 464, 190, 493, 137, 472, 475, 496, 118, 419, 423, 487, 317, 490, 19, 499, 91, 495, 125, 497, 210, 477, 90, 498, 401, 489, 67, 496, 135, 454, 140, 407, 381, 499, 353

In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

DB_PATH = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=DB_PATH,
    collection_name="korean_pdf"
)

In [6]:
vectorstore.similarity_search("구개음화", k=3)

[Document(id='af8f3f4b-1f65-4780-b824-97db1ddb8f5f', metadata={'creator': 'PScript5.dll Version 5.2.2', 'author': 'admin', 'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'source': 'datasets/한글맞춤법 표준어규정 해설.pdf', 'creationdate': '2018-12-20T09:36:29+09:00', 'total_pages': 264, 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'moddate': '2018-12-20T09:38:09+09:00', 'page': 23, 'page_label': '24'}, page_content='는 부사 파생 접미사가 결합하여 부사가 되었으므로 구개음화가 실현되었지만, ‘곧이\n어’는 형식 형태소가 아닌 실질 형태소 부사가 결합한 말이므로 구개음화가 실현되지 \n않는다. \n곧이[고지]: 곧-(어근)+-이(부사 파생 접미사)\n곧이어[고디어]: 곧(부사)+이어(부사)\n현재 표준어에서 구개음화는 형태소와 형태소가 결합할 때 일어나는 현상이다. 그러\n므로 ‘마디, 견디다’와 같이 하나의 형태소 내부에서는 구개음화가 일어나지 않는다.'),
 Document(id='d0878e68-91a5-49e5-b69b-24528e3f8f89', metadata={'creationdate': '2018-12-20T09:36:29+09:00', 'total_pages': 264, 'title': '<C6ED2DBEEEB9AEB1D4B9FCC7D8BCB35FB1B9BEEEBFF8C3D6C1BE5F76657232302DBCF6C1A4BABB2E687770>', 'author': 'admin', 'source': 'datasets/한글맞춤법 표준어규정 해설